# Mplus data preparation for complex-survey structural equation modelling

Prepares the analytic dataset for the Mplus complex-survey SEM of the
dual-mediator pathway from rest-activity rhythm metrics to depressive symptoms
and cognitive function. Reads `data/processed/8_nhanes_analytic.csv` and writes
a space-delimited, header-less data file (`nhanes_sem.dat`, missing = `-9999`), a
variable manifest (`nhanes_sem_vars.txt`), an RA-quintile companion file
(`nhanes_sem_raq.dat`, `nhanes_sem_raq_vars.txt`), a companion file carrying the
covariate-complete flag for the unadjusted models (`nhanes_sem_covok.dat`,
`nhanes_sem_covok_vars.txt`) and sha256 sidecars, all in `data/mplus/`.

## Z-standardisation

Continuous variables are z-standardised before being written. The reference
sample is the depression analytic universe (adults aged 20 and older with
non-missing exposure, both mediators, PHQ-9 total and the M1 covariate set):

$$z_{x,i} = \frac{x_i - \mu_{\text{ref}}}{\sigma_{\text{ref}}}, \quad \mu_{\text{ref}} = \frac{1}{n_{\text{ref}}}\sum_{j \in \text{ref}} x_j, \quad \sigma_{\text{ref}} = \sqrt{\frac{1}{n_{\text{ref}} - 1}\sum_{j \in \text{ref}} (x_j - \mu_{\text{ref}})^2}$$

Cognition domain scores are z-standardised within the 60+ subsample with the
same formula (reference restricted to participants aged 60 and older with valid
sub-test data; scores of 0 are valid results and are kept):

$$\mathrm{MEM\_Z} = z_{60+}(\text{memory\_score}), \quad \mathrm{FLUEN\_Z} = z_{60+}(\text{fluency\_score}), \quad \mathrm{SPEED\_Z} = z_{60+}(\text{speed\_score})$$

The composite cognition outcome `COG_Z` is the unweighted mean of the three
within-60+ z-scores, used as a sensitivity outcome in place of the latent
cognition factor:

$$\mathrm{COG\_Z} = \frac{\mathrm{MEM\_Z} + \mathrm{FLUEN\_Z} + \mathrm{SPEED\_Z}}{3}$$

PHQ-9 total is the sum of the nine DPQ items with refusal (code 7) and
don't-know (code 9) set to missing, requiring all nine items non-missing:

$$\mathrm{PHQ9\_total} = \sum_{k=1}^{9} \mathrm{DPQ}_k, \quad \text{undefined if any } \mathrm{DPQ}_k \text{ missing}$$

The four-year MEC sampling weight is the two-year weight divided by two
(NHANES pooled-cycles convention):

$$\mathrm{WT4YR} = \mathrm{WTMEC2YR} \,/\, 2$$

## Cognition domains

| Domain | Mplus name | Source column | NHANES origin                              |
|--------|-----------|---------------|--------------------------------------------|
| Memory | `MEM_Z`   | `memory_score`  | CERAD Word List total + Delayed Recall     |
| Verbal fluency | `FLUEN_Z` | `fluency_score` | Animal Fluency total (CFDAST)      |
| Processing speed | `SPEED_Z` | `speed_score`   | Digit Symbol Substitution (CFDDS)|

## Variable name table (Mplus 8-character limit)

| Mplus name | Source                                          |
|-----------|-------------------------------------------------|
| SEQN      | participant identifier                           |
| IV / RA / IS | rest-activity rhythm metrics (z)              |
| BA_ADV    | KDM_BA_advance (KDM-Biological Age advancement, z) |
| SII_LOG   | log SII (z)                                      |
| NLR_LOG   | log NLR (z)                                      |
| OBJ_MVPA  | objective MVPA min/day (z); in the file, used by no model |
| PHQ9      | PHQ-9 total (z)                                  |
| COG_Z     | composite cognition z (sensitivity outcome)      |
| MEM_Z     | memory_score z within 60+                        |
| FLUEN_Z   | fluency_score z within 60+                       |
| SPEED_Z   | speed_score z within 60+                         |
| AGE / BMI / EDU / WEAR_T / SLEEP_H | covariates (z); SEX on the 0/1 scale |
| VOLUME    | TOTAL_MIMS (z over all adults), volume-adjusted models |
| AGE60     | 1 if RIDAGEYR >= 60; SUBPOPULATION for the cognition models |
| IV_AGE / IS_AGE / RA_AGE | exposure x AGE products of the z-scores   |
| IV_Q      | IV quintile (1-5) within adults, skew sensitivity |
| WT4YR / STRATUM / PSU | NHANES complex-survey design variables |

## Step 0 — Imports and paths


In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import os as _os

def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = _os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE = find_root()
PROC = BASE / 'data' / 'processed'
MPLUS = BASE / 'data' / 'mplus'
MPLUS.mkdir(parents=True, exist_ok=True)

import sys
sys.path.insert(0, str(BASE / 'scripts'))
from population import ADULT_MIN_AGE, OLD_MIN_AGE   # analysis population: adults aged 20+

ANALYTIC = PROC / '8_nhanes_analytic.csv'
DAT_OUT  = MPLUS / 'nhanes_sem.dat'
VARS_OUT = MPLUS / 'nhanes_sem_vars.txt'
MISSING = -9999

df = pd.read_csv(ANALYTIC)
print(f'Loaded: {df.shape}')

## Step 1 — Derive outcomes (PHQ-9 and cognition domain z-scores)

PHQ-9 total: sum of the nine DPQ items with refusal (7) and don't-know (9) set
to missing. Cognition domain scores (`memory_score`, `fluency_score`,
`speed_score`) are read from the analytic file; scores of 0 are valid results of a
completed test and are kept (NHANES leaves non-administered tests blank)
before z-standardisation within the 60+ subsample.


In [ ]:
# PHQ-9 total
dpq_cols = [f'DPQ{i:03d}' for i in [10,20,30,40,50,60,70,80,90]]
phq = df[dpq_cols].replace({7: np.nan, 9: np.nan})
df['PHQ9_total'] = phq.sum(axis=1, min_count=9)

# Cognition domain scores from the analytic file
needed = ['memory_score', 'fluency_score', 'speed_score']
missing_cols = [c for c in needed if c not in df.columns]
if missing_cols:
    raise KeyError(
        f'Expected columns missing from 8_nhanes_analytic.csv: {missing_cols}. '
        'Verify that the upstream pipeline emits memory_score / fluency_score / '
        'speed_score in the analytic file.'
    )

elig_60 = df['RIDAGEYR'] >= 60
print('Cognition domain diagnostics within 60+ subsample (raw 0 -> NaN):')
for src_col, out_col in [('memory_score', 'memory_z'),
                          ('fluency_score', 'fluency_z'),
                          ('speed_score', 'speed_z')]:
    raw = df[src_col]
    n_zero  = ((df[src_col] == 0) & elig_60).sum()   # valid zeros, kept
    s = raw.loc[elig_60].dropna()
    mu = s.mean(); sd = s.std()
    df[out_col] = np.where(elig_60, (raw - mu) / sd, np.nan)
    print(f'  {src_col:14s}  n_zero(kept) = {n_zero:4d}   n_valid = {len(s):4d}   '
          f'mean = {mu:7.2f}   sd = {sd:7.2f}')

# Composite cognition z (sensitivity outcome): mean of the three z-scores, all three required
df['Cog_composite_z'] = df[['memory_z', 'fluency_z', 'speed_z']].mean(axis=1, skipna=False)
# WTMEC4YR is stored in the analytic file; checked here, not re-derived.
assert 'WTMEC4YR' in df.columns, 'WTMEC4YR missing from the analytic file'
assert np.allclose(df['WTMEC4YR'], df['WTMEC2YR'] / 2.0, equal_nan=True), \
    'stored WTMEC4YR does not match WTMEC2YR / 2 -- the file is stale'

print()
print(f'PHQ-9 complete                : n = {df["PHQ9_total"].notna().sum()}')
print(f'Cog composite (all 3 non-NaN) : n = {df["Cog_composite_z"].notna().sum()}')

## Step 2 — Define the standardisation reference (depression analytic universe)

Adults aged 20 and older (the analysis population; see `population.py`) with
non-missing IV, KDM_BA, SII, PHQ9_total and the
M1 covariates; the mean and SD of each continuous variable in this sample are
used for z-standardisation.


In [ ]:
COVS_RAW = ['RIDAGEYR','RIAGENDR','BMXBMI','DMDEDUC2','Wear_Time','SLD010H']
key      = ['IV','KDM_BA','SII','PHQ9_total'] + COVS_RAW

# Reference sample for every z-score: the depression analytic universe within
# the analysis population (adults aged ADULT_MIN_AGE and over).
ref = df[(df['RIDAGEYR'] >= ADULT_MIN_AGE) & df[key].notna().all(axis=1)].copy()
print(f'Standardisation reference (depression universe, aged {ADULT_MIN_AGE}+): n = {len(ref)}')

def z_stats(col):
    s = ref[col].dropna()
    return float(s.mean()), float(s.std())

Z_TARGETS = {
    'IV': 'IV', 'RA': 'RA', 'IS': 'IS',
    'KDM_BA_advance': 'KDM_BA_advance',
    'SII_log': 'SII_log', 'NLR_log': 'NLR_log',
    'obj_mvpa_min': 'obj_mvpa_min',
    'PHQ9_total': 'PHQ9_total',
    'Cog_composite_z': 'Cog_composite_z',
    'RIDAGEYR': 'RIDAGEYR', 'BMXBMI': 'BMXBMI',
    'DMDEDUC2': 'DMDEDUC2',
    'Wear_Time': 'Wear_Time', 'SLD010H': 'SLD010H',
}
z_params = {col: z_stats(col) for col in Z_TARGETS}
for col, (mu, sd) in z_params.items():
    print(f'  {col:18s}  mean = {mu:8.3f}   sd = {sd:8.3f}')

## Step 3 — Assemble the Mplus-ready frame

Z-standardises the continuous variables, adds the cognition indicators, VOLUME,
AGE60, the exposure x AGE products, IV_Q and the design variables, restricts to
adults and checks that every stratum keeps at least 2 PSUs.


In [ ]:
_ref = df[df['RIDAGEYR'] >= ADULT_MIN_AGE]
_vol_z = ((_ref['TOTAL_MIMS'].mean(), _ref['TOTAL_MIMS'].std())
          if 'TOTAL_MIMS' in df.columns else (0.0, 1.0))

def zify(series, mu, sd):
    """Apply z = (x - mu) / sd to a pandas Series."""
    return (series - mu) / sd

out = pd.DataFrame()
out['SEQN']    = df['SEQN'].astype('Int64')
out['IV']      = zify(df['IV'],             *z_params['IV'])
out['RA']      = zify(df['RA'],             *z_params['RA'])
out['IS']      = zify(df['IS'],             *z_params['IS'])
out['BA_ADV']  = zify(df['KDM_BA_advance'], *z_params['KDM_BA_advance'])
out['SII_LOG'] = zify(df['SII_log'],        *z_params['SII_log'])
out['NLR_LOG'] = zify(df['NLR_log'],        *z_params['NLR_log'])
out['OBJ_MVPA']= zify(df['obj_mvpa_min'],   *z_params['obj_mvpa_min'])
out['PHQ9']    = zify(df['PHQ9_total'],     *z_params['PHQ9_total'])
out['COG_Z']   = zify(df['Cog_composite_z'],*z_params['Cog_composite_z'])
# Cognition domain indicators (within-60+ z), used as latent factor indicators
out['MEM_Z']   = df['memory_z']
out['FLUEN_Z'] = df['fluency_z']
out['SPEED_Z'] = df['speed_z']
out['AGE']     = zify(df['RIDAGEYR'],       *z_params['RIDAGEYR'])
out['SEX']     = df['RIAGENDR']
out['BMI']     = zify(df['BMXBMI'],         *z_params['BMXBMI'])
out['EDU']     = zify(df['DMDEDUC2'],       *z_params['DMDEDUC2'])
out['WEAR_T']  = zify(df['Wear_Time'],      *z_params['Wear_Time'])
out['SLEEP_H'] = zify(df['SLD010H'],        *z_params['SLD010H'])

# Total daily activity volume (TOTAL_MIMS), z-scored with the mean and SD of
# all adults (not the depression reference); see 5_accelerometer_RAR.
out['VOLUME']  = zify(df['TOTAL_MIMS'], *_vol_z) if 'TOTAL_MIMS' in df.columns else np.nan

# AGE60 marks the design subpopulation for the cognitive module (NHANES
# 2011-2014 administers the cognitive battery at ages 60 and over) and is used
# as SUBPOPULATION in M1_cognition.inp. A survey subpopulation is defined by a
# design characteristic, not by outcome missingness, and FIML can then use
# participants with partial cognitive indicators. On this file, among
# participants aged 60+ with a valid IV and complete covariates (n = 2,929),
# 2,549 have all three cognitive indicators and 2,732 have at least one.
out['AGE60']   = (df['RIDAGEYR'] >= OLD_MIN_AGE).astype('int8')

# Exposure x age products on the z-scores, so the main effects are
# interpretable at the sample mean
for _x in ['IV', 'IS', 'RA']:
    out[f'{_x}_AGE'] = out[_x] * out['AGE']

# IV quintiles for the skew sensitivity analysis (Supplementary Table S19
# gives the skewness of IV and of BA_ADV on the current analytic file). MLR
# gives robust standard errors under non-normality but does not protect the
# point estimate from extreme observations, and IV and BA_ADV form the a1
# path. Quintiles are also comparable to Xu et al. (2023), who reported
# effects per 1-quantile increase. Computed within the analysis population,
# on observed values only.
_iv = df.loc[df['RIDAGEYR'] >= ADULT_MIN_AGE, 'IV']
out['IV_Q'] = pd.qcut(_iv, 5, labels=False).astype('float') + 1

out['WT4YR']   = df['WTMEC4YR']
out['STRATUM'] = df['SDMVSTRA'].astype('Int64')
out['PSU']     = df['SDMVPSU'].astype('Int64')
# ADULT20 marks the analysis population. The file keeps every adult aged 18
# and over so that the survey design is intact; the models select the
# population through SUBPOPULATION (ADULT20 EQ 1), not by deleting rows.
out['ADULT20'] = (df['RIDAGEYR'] >= ADULT_MIN_AGE).astype('int8')

out = out[df['RIDAGEYR'] >= 18].copy()
print(f'Adult observations written to .dat: n = {len(out)}; in the analysis population (ADULT20 = 1): {int(out.ADULT20.sum()):,}')
print(f'Total columns: {len(out.columns)}')
print(f'  design-eligible for the cognitive module (AGE60 = 1): n = {int(out.AGE60.sum()):,}')

# Restricting the file to adults (instead of SUBPOPULATION) requires every
# stratum to keep at least 2 PSUs: 29 strata and 61 PSUs on this file, at every
# restriction down to 60+ with valid IV.
_st = out.groupby('STRATUM')['PSU'].nunique()
assert (_st >= 2).all(), (
    'A stratum has fewer than 2 PSUs after restricting to adults; design-based '
    'variance cannot be estimated for it. Switch to SUBPOPULATION instead of '
    'deleting rows.')
print(f'  design check: {out.STRATUM.nunique()} strata, '
      f'{out.groupby(["STRATUM","PSU"]).ngroups} PSUs, none with <2 PSU per stratum')

out_filled = out.fillna(MISSING)

## Step 4 — Write `nhanes_sem.dat`

Integer format for SEQN, STRATUM, PSU and SEX; six decimals otherwise; missing = -9999.


In [ ]:
fmt = []
for col in out_filled.columns:
    if col in ('SEQN', 'STRATUM', 'PSU', 'SEX'):
        fmt.append('%d')
    else:
        fmt.append('%.6f')

with open(DAT_OUT, 'w') as f:
    for row in out_filled.itertuples(index=False):
        items = []
        for v, f_ in zip(row, fmt):
            if f_ == '%d':
                items.append(f_ % int(v))
            else:
                items.append(f_ % float(v))
        f.write(' '.join(items) + '\n')
print(f'Wrote {DAT_OUT}  ({DAT_OUT.stat().st_size / 1e6:.1f} MB, {len(out_filled)} rows)')

## Step 5 — Write `nhanes_sem_vars.txt`

Column order, standardisation means and SDs, and coding notes for the .dat file.


In [ ]:
out_col_map = {
    'IV':'IV','RA':'RA','IS':'IS','KDM_BA_advance':'BA_ADV',
    'SII_log':'SII_LOG','NLR_log':'NLR_LOG','obj_mvpa_min':'OBJ_MVPA',
    'PHQ9_total':'PHQ9','Cog_composite_z':'COG_Z','RIDAGEYR':'AGE',
    'BMXBMI':'BMI','DMDEDUC2':'EDU','Wear_Time':'WEAR_T','SLD010H':'SLEEP_H',
}
manifest = [
    f'# Mplus variable manifest for nhanes_sem.dat',
    f'# Generated: {pd.Timestamp.today():%Y-%m-%d}',
    f'# Missing-value code: {MISSING}',
    f'# n adult observations: {len(out_filled)}',
    f'',
    f'# Cognition domains (z within the 60+ subsample; scores of 0 are valid and kept):',
    f'#   MEM_Z   = z(memory_score)    CERAD Word List total',
    f'#   FLUEN_Z = z(fluency_score)   Animal Fluency total (CFDAST)',
    f'#   SPEED_Z = z(speed_score)     Digit Symbol Substitution (CFDDS)',
    f'',
    f'# Column order in nhanes_sem.dat:',
]
for i, col in enumerate(out_filled.columns, start=1):
    manifest.append(f'#   {i:2d}  {col}')
manifest.append('')
manifest.append(f'# Standardisation reference: depression analytic universe (adults aged {ADULT_MIN_AGE}+).')
manifest.append('# Continuous variables stored as z-scores using these (mean, sd):')
for src_col, (mu, sd) in z_params.items():
    manifest.append(f'#   {out_col_map[src_col]:8s}  source = {src_col:20s}  mean = {mu:10.4f}   sd = {sd:10.4f}')
manifest.append('# SEX kept on the 0/1 scale.')
manifest.append('# AGE60 = 1 if RIDAGEYR >= 60, the DESIGN eligibility group for the')
manifest.append('#   NHANES cognitive module. Used as SUBPOPULATION in M1_cognition.inp.')
manifest.append('#   Do NOT define that subpopulation by outcome missingness.')
manifest.append('# WT4YR, STRATUM, PSU kept on the original NHANES survey scale.')
manifest.append(f'# ADULT20 = 1 if RIDAGEYR >= {ADULT_MIN_AGE}: the analysis population. Every model selects it')
manifest.append('#   through SUBPOPULATION; the file keeps all adults aged 18+ for the survey design.')
VARS_OUT.write_text('\n'.join(manifest))
print(f'Wrote {VARS_OUT}')
print('\n'.join(manifest[:25]))

## Step 6 — Sanity check

Prints the first rows of the .dat file, the 60+ cognition z-distributions and the SEX value counts.


In [ ]:
with open(DAT_OUT) as f:
    for i, line in enumerate(f):
        if i >= 3: break
        print(line.rstrip())

print('\nCognition domain z-distributions in the 60+ subsample:')
elig = df[df['RIDAGEYR'] >= 60]
for c in ['memory_z', 'fluency_z', 'speed_z']:
    s = elig[c].dropna()
    print(f'  {c:10s}  n_valid = {len(s):4d}   mean = {s.mean():.3f}   sd = {s.std():.3f}   '
          f'min/max = {s.min():.2f} / {s.max():.2f}')

import collections
with open(DAT_OUT) as f:
    sex_idx = list(out_filled.columns).index('SEX')
    counts = collections.Counter()
    for line in f:
        counts[line.split()[sex_idx]] += 1
print('\nSEX value distribution in .dat:')
for k, v in counts.most_common():
    print(f'  {k}: {v}')

## Step 7 — RA-quintile file and data checksums

Writes `nhanes_sem_raq.dat` (nhanes_sem.dat plus RA quintiles, for the
non-linearity sensitivity in script 13) and sha256 sidecars for the .dat files.


In [ ]:
# RA quintiles for the non-linearity sensitivity in script 13: the RA-outcome
# association is non-linear for both outcomes (Wald p < 0.001; signal
# concentrated above RA ~ 0.7-0.8), while IV and IS are linear.
# RA_Q is written to a separate file so that the NAMES lists of the existing
# Mplus models are unchanged. Quintiles are defined within the analysis
# population (ADULT20 = 1) on observed values.
import re as _re
import hashlib as _hashlib

_manifest = [m.group(1) for m in _re.finditer(
    r'^#\s+\d+\s+(\w+)\s*$', (MPLUS / 'nhanes_sem_vars.txt').read_text(), _re.M)]
_dat = pd.read_csv(MPLUS / 'nhanes_sem.dat', sep=r'\s+', header=None,
                   names=_manifest, na_values=-9999)
# RA in the .dat is z-scored; quintiles are invariant to a monotone transform.
_dat['RA_Q'] = np.nan
_in = _dat['ADULT20'] == 1
_dat.loc[_in, 'RA_Q'] = pd.qcut(_dat.loc[_in, 'RA'], 5, labels=False).astype('float') + 1
_out = _dat.fillna(-9999)
_out.to_csv(MPLUS / 'nhanes_sem_raq.dat', sep=' ', header=False, index=False,
            float_format='%.6f')
with open(MPLUS / 'nhanes_sem_raq_vars.txt', 'w') as f:
    f.write('# nhanes_sem_raq.dat -- nhanes_sem.dat plus RA quintiles\n')
    for i, c in enumerate(_manifest + ['RA_Q'], 1):
        f.write(f'#  {i:2d}  {c}\n')
print(f'wrote nhanes_sem_raq.dat: {len(_out):,} rows x {len(_out.columns)} cols '
      f'(RA_Q missing for {int((_out.RA_Q == -9999).sum()):,})')

# sha256 sidecars: the hash, not the mtime, certifies that the Mplus .out files
# correspond to this .dat.
for _f in ('nhanes_sem.dat', 'nhanes_sem_raq.dat', 'nhanes_sem_trimmed.dat'):
    _p = MPLUS / _f
    if _p.exists():
        _h = _hashlib.sha256(_p.read_bytes()).hexdigest()
        (MPLUS / (_f + '.sha256')).write_text(_h + '\n')
        print(f'  sha256 {_f:26s} {_h[:16]}...')


## Step 8 — Covariate-complete flag file for the unadjusted models

The unadjusted counterparts of the primary models (STROBE item 16a;
`9_mplus_models/build_inputs.py` writes them as `m_*_crude.inp`) contain no
covariates, so Mplus would no longer drop participants with a missing
covariate. `COVOK` marks the participants with all six covariates observed;
the unadjusted inputs add `COVOK EQ 1` to the `SUBPOPULATION` statement that
defines the age cell (Mplus does not allow `USEOBSERVATIONS` together with
`SUBPOPULATION`). Each unadjusted model therefore has the same n as its
adjusted counterpart, which `mplus_report` asserts.


In [ ]:
COVS_DAT = ['AGE', 'SEX', 'BMI', 'EDU', 'WEAR_T', 'SLEEP_H']
_dat['COVOK'] = _dat[COVS_DAT].notna().all(axis=1).astype(int)
_out = _dat.drop(columns=['RA_Q']).fillna(-9999)
_out.to_csv(MPLUS / 'nhanes_sem_covok.dat', sep=' ', header=False, index=False,
            float_format='%.6f')
with open(MPLUS / 'nhanes_sem_covok_vars.txt', 'w') as f:
    f.write('# nhanes_sem_covok.dat -- nhanes_sem.dat plus COVOK, 1 if all six '
            'covariates (AGE SEX BMI EDU WEAR_T SLEEP_H) are observed\n')
    for i, c in enumerate(_manifest + ['COVOK'], 1):
        f.write(f'#  {i:2d}  {c}\n')
_h = _hashlib.sha256((MPLUS / 'nhanes_sem_covok.dat').read_bytes()).hexdigest()
(MPLUS / 'nhanes_sem_covok.dat.sha256').write_text(_h + '\n')
print(f'wrote nhanes_sem_covok.dat: {len(_out):,} rows x {len(_out.columns)} cols; '
      f'COVOK = 1 for {int(_out.COVOK.sum()):,}')
print(f'  sha256 nhanes_sem_covok.dat      {_h[:16]}...')